# Lab 02-1：股票与 ETF 标的行业分类

**实验目标**：形成可复查的 A 股行业分类与 ETF 行业初筛流程，为后续行业市值排名和
走势相关性研究提供输入。

数据截止日为 **2026-07-29（最后完整交易日）**。本 Notebook 不使用 2026-07-30
盘中价格作为市值排名依据。


## 口径与数据路由

| 对象 | 主分类依据 | 行情/市值 | 校验与边界 |
| --- | --- | --- | --- |
| A 股 | 申万一级行业指数当前成分 | 腾讯财经实时行情 | 总市值按昨收回推至截止日；深交所官方行业仅作跨分类体系对照 |
| ETF | 东财 ETF 清单与基金类型 | 东财/同花顺 ETF 清单互查 | 名称关键词只是初筛；权威结论仍需基金合同、招募说明书或跟踪指数说明 |

申万一级行业是一套互斥的投资研究分类；交易所行业是监管口径，两者粒度不同，
不应要求名称逐字一致。ETF 是基金产品，不天然等于上市公司的“所属行业”：
宽基、策略、债券、商品、跨境 ETF 应标为非行业 ETF。


In [1]:
import json
import math
import time
import urllib.request
from pathlib import Path

import akshare as ak
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

WORKING_DIR = Path.cwd().resolve()
LAB_DIR = next(
    (
        p
        for p in [WORKING_DIR, *WORKING_DIR.parents]
        if p.name == "02_行业走势相关性研究"
    ),
    None,
)
if LAB_DIR is None:
    candidate = WORKING_DIR / "labs" / "02_行业走势相关性研究"
    if candidate.is_dir():
        LAB_DIR = candidate
if LAB_DIR is None:
    raise FileNotFoundError("请从仓库根目录或 labs/02_行业走势相关性研究 运行")

DATA_DIR = LAB_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = pd.Timestamp("2026-07-29")
RUN_DATE = pd.Timestamp.now(tz="Asia/Shanghai")
print("AKShare:", ak.__version__)
print("运行时间:", RUN_DATE)
print("市值快照截止日:", SNAPSHOT_DATE.date())


AKShare: 1.18.78
运行时间: 2026-07-30 14:43:24.940593+08:00
市值快照截止日: 2026-07-29


In [2]:
def normalize_code(value) -> str:
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text.zfill(6)


def tencent_quote(codes: list[str], batch_size: int = 80) -> pd.DataFrame:
    """批量获取腾讯行情；串行小批次，返回当前价、昨收与总市值。"""
    rows = []
    for start in range(0, len(codes), batch_size):
        batch = codes[start : start + batch_size]
        prefixed = [
            ("sh" if c.startswith(("6", "9")) else "bj" if c.startswith("8") else "sz") + c
            for c in batch
        ]
        url = "https://qt.gtimg.cn/q=" + ",".join(prefixed)
        last_error = None
        for attempt in range(3):
            try:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "Mozilla/5.0"}
                )
                with urllib.request.urlopen(request, timeout=20) as response:
                    payload = response.read().decode("gbk", errors="replace")
                break
            except Exception as exc:
                last_error = exc
                time.sleep(1.0 * (attempt + 1))
        else:
            raise RuntimeError(f"腾讯行情批次获取失败: {last_error}")

        for line in payload.strip().split(";"):
            if "=" not in line or '"' not in line:
                continue
            key = line.split("=", 1)[0].split("_")[-1]
            values = line.split('"')[1].split("~")
            if len(values) < 46:
                continue
            def number(index: int) -> float:
                try:
                    return float(values[index]) if values[index] else np.nan
                except (ValueError, IndexError):
                    return np.nan
            rows.append(
                {
                    "symbol": key[2:],
                    "quote_name": values[1],
                    "price": number(3),
                    "last_close": number(4),
                    "mcap_current_yi": number(44),
                    "float_mcap_current_yi": number(45),
                }
            )
        time.sleep(0.05)
    frame = pd.DataFrame(rows).drop_duplicates("symbol", keep="last")
    return frame


def safe_to_parquet(frame: pd.DataFrame, path: Path) -> None:
    frame.to_parquet(path, index=False)
    assert path.is_file()


## 1. A 股：申万一级行业成分

先获取 31 个申万一级行业，再逐行业获取当前成分。申万成分与腾讯行情分别来自不同
数据提供方；行业归属与市值因此不是同一接口内的自我验证。


In [3]:
sw_realtime = ak.index_realtime_sw(symbol="一级行业").copy()
sw_industries = (
    sw_realtime[["指数代码", "指数名称"]]
    .rename(columns={"指数代码": "industry_code", "指数名称": "industry_name"})
    .drop_duplicates("industry_code")
    .sort_values("industry_code")
    .reset_index(drop=True)
)

component_frames = []
component_errors = []
for row in sw_industries.itertuples(index=False):
    try:
        part = ak.index_component_sw(symbol=row.industry_code).copy()
        part = part.rename(
            columns={
                "证券代码": "symbol",
                "证券名称": "name",
                "最新权重": "latest_weight_pct",
                "计入日期": "included_date",
            }
        )
        part["symbol"] = part["symbol"].map(normalize_code)
        part["industry_code"] = row.industry_code
        part["industry_name"] = row.industry_name
        component_frames.append(
            part[
                [
                    "symbol",
                    "name",
                    "industry_code",
                    "industry_name",
                    "latest_weight_pct",
                    "included_date",
                ]
            ]
        )
    except Exception as exc:
        component_errors.append(
            {"industry_code": row.industry_code, "error": repr(exc)}
        )
    time.sleep(0.10)

if component_errors:
    display(pd.DataFrame(component_errors))
    raise RuntimeError("申万行业成分存在获取失败，停止生成不完整分类")

stock_classification = pd.concat(component_frames, ignore_index=True)
stock_classification["classification_system"] = "申万一级行业"
stock_classification["classification_source"] = "申万指数（AKShare index_component_sw）"
stock_classification["snapshot_date"] = SNAPSHOT_DATE

duplicate_assignments = (
    stock_classification.groupby("symbol")["industry_code"].nunique().gt(1)
)
duplicate_symbols = duplicate_assignments[duplicate_assignments].index.tolist()
print("申万一级行业数:", len(sw_industries))
print("行业成分记录数:", len(stock_classification))
print("唯一股票数:", stock_classification["symbol"].nunique())
print("跨一级行业重复股票数:", len(duplicate_symbols))
display(sw_industries)


  0%|          | 0/1 [00:00<?, ?it/s]

申万一级行业数: 31
行业成分记录数: 5200
唯一股票数: 5200
跨一级行业重复股票数: 0


,industry_code,industry_name
0,801010,农林牧渔
1,801030,基础化工
2,801040,钢铁
3,801050,有色金属
4,801080,电子
5,801110,家用电器
6,801120,食品饮料
7,801130,纺织服饰
8,801140,轻工制造
9,801150,医药生物


In [4]:
# 腾讯实时总市值按 昨收/现价 回推至 2026-07-29 收盘。
unique_symbols = sorted(stock_classification["symbol"].unique())
quotes = tencent_quote(unique_symbols)
quotes["mcap_asof_yi"] = np.where(
    quotes["price"].gt(0) & quotes["last_close"].gt(0),
    quotes["mcap_current_yi"] * quotes["last_close"] / quotes["price"],
    quotes["mcap_current_yi"],
)
quotes["mcap_method"] = "腾讯当前总市值×昨收/现价"
quotes["snapshot_date"] = SNAPSHOT_DATE

stock_classification = stock_classification.merge(
    quotes[
        [
            "symbol",
            "quote_name",
            "price",
            "last_close",
            "mcap_current_yi",
            "mcap_asof_yi",
            "mcap_method",
        ]
    ],
    on="symbol",
    how="left",
    validate="many_to_one",
)
stock_classification["quote_available"] = (
    stock_classification["mcap_asof_yi"].notna()
    & stock_classification["mcap_asof_yi"].gt(0)
)
stock_classification["name_match"] = (
    stock_classification["name"].str.replace(" ", "", regex=False)
    == stock_classification["quote_name"].fillna("").str.replace(" ", "", regex=False)
)

industry_market_cap = (
    stock_classification.groupby(
        ["industry_code", "industry_name"], as_index=False
    )
    .agg(
        constituent_count=("symbol", "nunique"),
        quote_count=("quote_available", "sum"),
        total_mcap_yi=("mcap_asof_yi", "sum"),
    )
)
industry_market_cap["quote_coverage"] = (
    industry_market_cap["quote_count"]
    / industry_market_cap["constituent_count"]
)
industry_market_cap = industry_market_cap.sort_values(
    "total_mcap_yi", ascending=False
).reset_index(drop=True)
industry_market_cap["rank"] = np.arange(1, len(industry_market_cap) + 1)
industry_market_cap["snapshot_date"] = SNAPSHOT_DATE

quality_summary = pd.DataFrame(
    [
        ("申万一级行业数", len(sw_industries)),
        ("成分记录数", len(stock_classification)),
        ("唯一股票数", stock_classification["symbol"].nunique()),
        ("跨一级行业重复股票数", len(duplicate_symbols)),
        ("腾讯市值覆盖率", stock_classification["quote_available"].mean()),
        ("腾讯名称一致率（有报价）", stock_classification.loc[
            stock_classification["quote_available"], "name_match"
        ].mean()),
    ],
    columns=["quality_metric", "value"],
)

safe_to_parquet(stock_classification, DATA_DIR / "stock_industry_classification.parquet")
safe_to_parquet(quotes, DATA_DIR / "tencent_quote_snapshot.parquet")
industry_market_cap.to_csv(
    DATA_DIR / "industry_market_cap.csv", index=False, encoding="utf-8-sig"
)
quality_summary.to_csv(
    DATA_DIR / "classification_quality.csv", index=False, encoding="utf-8-sig"
)
display(quality_summary)
display(industry_market_cap.head(10))


,quality_metric,value
0,申万一级行业数,31.000000
1,成分记录数,5200.000000
2,唯一股票数,5200.000000
3,跨一级行业重复股票数,0.000000
4,腾讯市值覆盖率,1.000000
5,腾讯名称一致率（有报价）,0.986923


,industry_code,industry_name,constituent_count,quote_count,total_mcap_yi,quote_coverage,rank,snapshot_date
0,801080,电子,489,489,149238.320419,1.0,1,2026-07-29
1,801780,银行,42,42,102498.794976,1.0,2,2026-07-29
2,801730,电力设备,377,377,67455.451214,1.0,3,2026-07-29
3,801790,非银金融,79,79,55446.671574,1.0,4,2026-07-29
4,801150,医药生物,478,478,54027.361807,1.0,5,2026-07-29
5,801890,机械设备,535,535,47697.138598,1.0,6,2026-07-29
6,801770,通信,122,122,43817.673747,1.0,7,2026-07-29
7,801050,有色金属,140,140,42080.548545,1.0,8,2026-07-29
8,801030,基础化工,410,410,39456.171506,1.0,9,2026-07-29
9,801120,食品饮料,122,122,37405.921404,1.0,10,2026-07-29


## 2. 交易所行业口径对照

深交所公开 A 股列表带有证监会大类行业字段。它与申万一级行业的研究口径不同，
此处只检查代码连接是否合理并展示映射关系，不用它覆盖申万分类。


In [5]:
try:
    szse = ak.stock_info_sz_name_code(symbol="A股列表").copy()
    szse_crosscheck = szse.rename(
        columns={
            "A股代码": "symbol",
            "A股简称": "exchange_name",
            "所属行业": "exchange_industry",
        }
    )[["symbol", "exchange_name", "exchange_industry"]]
    szse_crosscheck["symbol"] = szse_crosscheck["symbol"].map(normalize_code)
    szse_crosscheck = szse_crosscheck.merge(
        stock_classification[
            ["symbol", "industry_code", "industry_name"]
        ].drop_duplicates("symbol"),
        on="symbol",
        how="left",
        validate="one_to_one",
    )
    szse_crosscheck["sw_available"] = szse_crosscheck["industry_code"].notna()
    szse_crosscheck.to_csv(
        DATA_DIR / "exchange_industry_crosscheck.csv",
        index=False,
        encoding="utf-8-sig",
    )
    print("深交所 A 股代码数:", len(szse_crosscheck))
    print("可连接到申万一级行业比例:", f"{szse_crosscheck['sw_available'].mean():.2%}")
    display(
        pd.crosstab(
            szse_crosscheck["exchange_industry"],
            szse_crosscheck["industry_name"],
        ).head(10)
    )
except Exception as exc:
    szse_crosscheck = pd.DataFrame()
    print("深交所行业对照获取失败，不影响申万主分类:", repr(exc))


深交所 A 股代码数: 2893
可连接到申万一级行业比例: 99.97%


industry_name,交通运输,传媒,公用事业,农林牧渔,医药生物,商贸零售,国防军工,基础化工,家用电器,建筑材料,建筑装饰,房地产,有色金属,机械设备,汽车,煤炭,环保,电力设备,电子,石油石化,社会服务,纺织服饰,综合,美容护理,计算机,轻工制造,通信,钢铁,银行,非银金融,食品饮料
exchange_industry,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
A 农林牧渔,0,0,0,27,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
B 采矿业,0,0,1,0,0,0,0,1,0,0,0,0,14,1,0,5,0,0,0,7,0,0,0,0,0,0,0,3,0,0,0
C 制造业,0,1,6,36,202,1,69,235,59,45,6,1,63,304,148,1,29,220,242,15,3,52,2,17,50,91,55,16,0,1,55
D 水电煤气,0,0,50,0,0,0,0,0,0,0,0,0,0,1,0,0,8,0,0,0,0,0,1,0,0,0,0,0,0,0,0
E 建筑业,0,0,0,0,1,0,0,0,0,0,43,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
F 批发零售,2,0,0,2,21,33,0,4,0,0,1,0,0,1,1,0,1,2,12,4,0,6,5,0,2,1,0,0,0,0,2
G 运输仓储,35,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
H 住宿餐饮,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0
I 信息技术,0,40,0,0,0,5,5,0,0,0,0,0,0,5,1,0,0,3,2,0,3,0,1,0,165,0,26,0,0,1,0


## 3. ETF：双清单与名称初筛

ETF 主表来自东财，独立清单来自同花顺。行业初筛按基金简称关键词映射到申万一级行业；
结果明确保留 `classification_confidence` 和 `needs_prospectus_review`。这一步适合构造
待核验候选池，不适合直接作为投资研究的最终行业事实。


In [6]:
try:
    etf_em_raw = ak.fund_etf_spot_em().copy()
    etf_em = etf_em_raw.rename(
        columns={
            "代码": "symbol",
            "名称": "name_em",
            "总市值": "market_cap_yuan",
            "数据日期": "data_date",
        }
    )[["symbol", "name_em", "market_cap_yuan", "data_date"]]
    etf_em["symbol"] = etf_em["symbol"].map(normalize_code)
    time.sleep(1.5)
except Exception as exc:
    etf_em = pd.DataFrame(
        columns=["symbol", "name_em", "market_cap_yuan", "data_date"]
    )
    print("东财 ETF 清单获取失败，降级为同花顺:", repr(exc))

etf_ths_raw = ak.fund_etf_spot_ths().copy()
etf_ths = etf_ths_raw.rename(
    columns={
        "基金代码": "symbol",
        "基金名称": "name_ths",
        "基金类型": "fund_type_ths",
        "查询日期": "query_date_ths",
    }
)[["symbol", "name_ths", "fund_type_ths", "query_date_ths"]]
etf_ths["symbol"] = etf_ths["symbol"].map(normalize_code)

if etf_em.empty:
    etf_universe = etf_ths.copy()
    etf_universe["name_em"] = pd.NA
    etf_universe["market_cap_yuan"] = np.nan
    etf_universe["data_date"] = pd.NaT
else:
    etf_universe = etf_em.merge(
        etf_ths, on="symbol", how="outer", validate="one_to_one"
    )

etf_universe["name"] = etf_universe["name_em"].fillna(
    etf_universe["name_ths"]
)
etf_universe["source_count"] = (
    etf_universe["name_em"].notna().astype(int)
    + etf_universe["name_ths"].notna().astype(int)
)
etf_universe["name_match"] = (
    etf_universe["name_em"].fillna("").str.replace(" ", "", regex=False)
    == etf_universe["name_ths"].fillna("").str.replace(" ", "", regex=False)
)

INDUSTRY_KEYWORDS = [
    ("银行", ["银行"]),
    ("非银金融", ["证券", "券商", "保险", "非银金融"]),
    ("电子", ["半导体", "芯片", "电子", "消费电子"]),
    ("计算机", ["计算机", "软件", "云计算", "信创", "人工智能", "AI"]),
    ("通信", ["通信", "5G"]),
    ("食品饮料", ["食品饮料", "白酒", "酒ETF", "食品"]),
    ("医药生物", ["医药", "医疗", "生物", "创新药", "中药"]),
    ("电力设备", ["电力设备", "光伏", "电池", "新能源车"]),
    ("汽车", ["汽车", "智能车"]),
    ("房地产", ["房地产", "地产"]),
    ("煤炭", ["煤炭"]),
    ("钢铁", ["钢铁"]),
    ("有色金属", ["有色", "稀土", "黄金", "金属"]),
    ("国防军工", ["军工", "国防", "航天", "航空"]),
    ("传媒", ["传媒", "游戏", "动漫"]),
    ("美容护理", ["美容", "美妆"]),
    ("社会服务", ["旅游", "酒店", "教育", "社会服务"]),
    ("家用电器", ["家电"]),
    ("农林牧渔", ["农业", "养殖", "畜牧", "农林牧渔"]),
    ("建筑材料", ["建筑材料", "建材"]),
    ("建筑装饰", ["建筑装饰", "基建"]),
    ("交通运输", ["交通运输", "物流", "机场", "航运"]),
    ("机械设备", ["机械", "机器人", "机床"]),
    ("石油石化", ["石油石化", "油气"]),
    ("基础化工", ["基础化工", "化工"]),
    ("纺织服饰", ["纺织", "服装"]),
    ("商贸零售", ["商贸零售", "零售"]),
    ("环保", ["环保"]),
    ("公用事业", ["公用事业", "电力ETF", "绿电"]),
    ("轻工制造", ["轻工", "家居"]),
    ("综合", ["综合行业"]),
]
NON_INDUSTRY_KEYWORDS = [
    "沪深300", "中证500", "上证50", "创业板", "科创", "红利",
    "债", "货币", "黄金ETF", "商品", "纳指", "标普", "恒生", "港股",
    "日经", "德国", "法国", "沙特", "东南亚", "REIT",
]

def classify_etf_name(name: str) -> tuple[str, str, bool]:
    text = str(name)
    for industry, keywords in INDUSTRY_KEYWORDS:
        if any(keyword in text for keyword in keywords):
            return industry, "名称关键词初筛", True
    if any(keyword in text for keyword in NON_INDUSTRY_KEYWORDS):
        return "非行业ETF", "名称规则", False
    return "待核验", "无可靠结构化跟踪指数", True

classified = etf_universe["name"].map(classify_etf_name)
etf_universe[["industry_initial", "classification_method", "needs_prospectus_review"]] = (
    pd.DataFrame(classified.tolist(), index=etf_universe.index)
)
etf_universe["classification_confidence"] = np.where(
    etf_universe["industry_initial"].eq("非行业ETF"),
    "中",
    np.where(etf_universe["industry_initial"].eq("待核验"), "低", "低"),
)
etf_universe["snapshot_date"] = SNAPSHOT_DATE
etf_universe = etf_universe.sort_values(
    ["industry_initial", "market_cap_yuan"], ascending=[True, False]
)
safe_to_parquet(etf_universe, DATA_DIR / "etf_industry_classification.parquet")

etf_summary = (
    etf_universe.groupby("industry_initial", dropna=False)
    .agg(
        etf_count=("symbol", "nunique"),
        dual_source_count=("source_count", lambda x: int((x == 2).sum())),
        market_cap_yi=("market_cap_yuan", lambda x: x.sum(min_count=1) / 1e8),
    )
    .sort_values("etf_count", ascending=False)
)
print("ETF 唯一代码数:", etf_universe["symbol"].nunique())
print("双清单同时覆盖比例:", f"{etf_universe['source_count'].eq(2).mean():.2%}")
print("需基金文件进一步核验比例:", f"{etf_universe['needs_prospectus_review'].mean():.2%}")
display(etf_summary.head(15))

sample_codes = ["510300", "510500", "512800", "512480", "512170"]
display(
    etf_universe.loc[
        etf_universe["symbol"].isin(sample_codes),
        [
            "symbol",
            "name",
            "fund_type_ths",
            "industry_initial",
            "classification_method",
            "classification_confidence",
            "needs_prospectus_review",
            "source_count",
        ],
    ].sort_values("symbol")
)


  0%|          | 0/15 [00:00<?, ?it/s]

ETF 唯一代码数: 1656
双清单同时覆盖比例: 93.96%
需基金文件进一步核验比例: 68.30%


,etf_count,dual_source_count,market_cap_yi
industry_initial,,,
非行业ETF,525,465,15185.037745
待核验,516,504,8910.150726
医药生物,102,99,2345.174165
计算机,80,76,953.985559
电子,57,57,3058.263134
有色金属,53,52,3216.246491
电力设备,51,46,489.300383
机械设备,33,32,581.924012
非银金融,29,28,1915.676637


,symbol,name,fund_type_ths,industry_initial,classification_method,classification_confidence,needs_prospectus_review,source_count
749,510300,沪深300ETF华泰柏瑞,股票型,非行业ETF,名称规则,中,False,2
759,510500,中证500ETF南方,股票型,非行业ETF,名称规则,中,False,2
848,512170,医疗ETF华宝,股票型,医药生物,名称关键词初筛,低,True,2
868,512480,半导体ETF国联安,股票型,电子,名称关键词初筛,低,True,2
893,512800,银行ETF华宝,股票型,银行,名称关键词初筛,低,True,2


In [7]:
manifest = {
    "schema_version": "1.0",
    "run_time": RUN_DATE.isoformat(),
    "snapshot_date": SNAPSHOT_DATE.date().isoformat(),
    "stock_classification": {
        "system": "申万一级行业",
        "source": "申万指数 via AKShare",
        "industry_count": int(len(sw_industries)),
        "record_count": int(len(stock_classification)),
        "unique_stock_count": int(stock_classification["symbol"].nunique()),
        "duplicate_assignment_count": int(len(duplicate_symbols)),
        "quote_coverage": float(stock_classification["quote_available"].mean()),
    },
    "market_cap": {
        "source": "腾讯财经",
        "method": "当前总市值×昨收/现价，回推至最后完整交易日",
        "unit": "亿元人民币",
    },
    "etf_classification": {
        "sources": ["东方财富", "同花顺"],
        "method": "名称关键词初筛；最终需基金合同/招募说明书/跟踪指数说明",
        "unique_etf_count": int(etf_universe["symbol"].nunique()),
        "dual_source_ratio": float(etf_universe["source_count"].eq(2).mean()),
    },
}
(DATA_DIR / "classification_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

assert len(sw_industries) == 31
assert not stock_classification.duplicated(
    ["symbol", "industry_code"]
).any()
assert stock_classification["quote_available"].mean() >= 0.95
assert industry_market_cap["total_mcap_yi"].gt(0).all()
assert etf_universe["symbol"].is_unique
print("分类数据质量门禁通过")
print(f"数据目录: {DATA_DIR.relative_to(LAB_DIR.parents[1])}")


分类数据质量门禁通过
数据目录: labs\02_行业走势相关性研究\data


## 本 Notebook 的结论边界

1. A 股行业分类可稳定地按“申万一级行业成分”构造，并用独立腾讯行情补足市值。
2. 交易所行业与申万行业是两套分类体系；对照的目的在于发现错码/漏码，不是强制同名。
3. ETF 的基金类型与行业暴露不是同一个字段。名称规则只能初筛，正式研究必须核验跟踪指数
   及基金法律文件；宽基、策略、债券、商品、跨境 ETF 不应硬塞进 A 股行业。
4. 市值排名是 2026-07-29 的静态快照，用它回看历史会产生选择时点偏差；相关性结果应
   解读为“当前大行业的历史联动”，而非当时可实时构建的无偏策略。
